In [1]:
#1 Load libraries
#2 Load model
#3 Load dataset
#4 Helper functions
#5 Extractor functions
#6 Prompt functions
#7 Inference function (공통)
#8 Zero-shot experiment
#9 CoT experiment
#10 Few-shot experiment
#11 Save results

In [2]:
#기본 라이브러리
import os
import torch
import json
import re
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [3]:
# Result path 설정
BASE_PATH = "/gpfs/data/oermannlab/gaifl/users/yl14814/SLM_hybrid_project"

MODEL_NAME = "llama31"
DATASET_NAME = "MedQA"

BASELINE_RESULT_DIR = f"{BASE_PATH}/03_results/baselines/{MODEL_NAME}/{DATASET_NAME}"
DEBUG_RESULT_DIR = f"{BASE_PATH}/03_results/debug/{MODEL_NAME}/{DATASET_NAME}"

import os
os.makedirs(BASELINE_RESULT_DIR, exist_ok=True)
os.makedirs(DEBUG_RESULT_DIR, exist_ok=True)

print("Baseline path:", BASELINE_RESULT_DIR)
print("Debug path:", DEBUG_RESULT_DIR)

Baseline path: /gpfs/data/oermannlab/gaifl/users/yl14814/SLM_hybrid_project/03_results/baselines/llama31/MedQA
Debug path: /gpfs/data/oermannlab/gaifl/users/yl14814/SLM_hybrid_project/03_results/debug/llama31/MedQA


In [4]:
#Llama-3.1 Load
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded


In [5]:
import torch
print(torch.cuda.is_available())
print(model.device)

True
cuda:0


In [6]:
#BigBio MedQA Load
#Data Load
dataset = load_dataset(
    "bigbio/med_qa",
    "med_qa_en_4options_bigbio_qa",
    trust_remote_code=True
)

test_data = dataset["test"]

print("Test size:", len(test_data))
print(test_data[0])

Test size: 1273
{'id': '0', 'question_id': '0', 'document_id': '0', 'question': 'A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?', 'type': 'multiple_choice', 'choices': ['Disclose the error to the patient and put it in the operative report', 'Tell the attending that he cannot fail to disclose this mistake', 'Report the physician to the ethics committee', 'Refuse to dictate the operative report'], 'context': '', 'answer': ['Tell the att

In [7]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

In [8]:
# Helper Functions
# dataset utility
def get_choices(example):

    if isinstance(example["choices"], dict):
        return example["choices"]["text"]

    if isinstance(example["choices"], list):
        return example["choices"]

    return []

In [9]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

In [10]:
# Helper Functions
# gold->A/B/C/D translate(변환)
def get_gold_letter(example):

    choices = get_choices(example)

    labels = ["A","B","C","D"]

    # answer가 text일 때
    if isinstance(example["answer"], list):

        gold_text = example["answer"][0].strip().lower()

        for i, choice in enumerate(choices):

            if choice.strip().lower() == gold_text:
                return labels[i]

    # answer가 letter일 때
    if isinstance(example["answer"], str):

        ans = example["answer"].strip().upper()

        if ans in ["A","B","C","D"]:
            return ans

    return "INVALID"

In [11]:
# Extra Function
# Zero-shot / Few-shot extractor
def extract_choice(text):

    text = text.strip()

    # Answer: B
    match = re.search(r"answer\s*[:\-]?\s*(A|B|C|D)", text, re.IGNORECASE)
    if match:
        return match.group(1)

    # first letter
    match = re.match(r"^\s*(A|B|C|D)[\.\)]?", text)
    if match:
        return match.group(1)

    return "INVALID"

In [12]:
# Extra Function
# CoT extractor
def extract_cot_answer(text):

    text = text.strip()

    patterns = [
        r"final answer\s*[:\-]?\s*(A|B|C|D)",
        r"answer\s*[:\-]?\s*(A|B|C|D)",
        r"correct option\s*(A|B|C|D)",
        r"option\s*(A|B|C|D)",
        r"\b(A|B|C|D)\b"
    ]

    for p in patterns:
        match = re.search(p, text, re.IGNORECASE)
        if match:
            return match.group(1).upper()

    return "INVALID"

In [13]:
#Zero-shot_Chat Template Prompt 
def build_zero_shot_prompt(example):

    question = example["question"]
    choices = get_choices(example)

    labels = ["A", "B", "C", "D"]

    options_text = "\n".join(
        [f"{labels[i]}. {choice}" for i, choice in enumerate(choices)]
    )

    prompt = f"""
You are a medical expert solving a USMLE clinical question.

Question:
{question}

Options:
{options_text}

Choose the correct answer.

Answer:
"""

    return prompt

In [14]:
#CoT Prompt
def build_cot_prompt(example):

    question = example["question"]
    choices = get_choices(example)

    options_text = "\n".join(
        [f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)]
    )

    prompt = f"""
You are a medical expert answering a USMLE clinical question.

Question:
{question}

Options:
{options_text}

Think through the problem step by step.

At the end of your reasoning, output ONLY the letter of the correct answer.

Format strictly as:

Final Answer: X

Where X is one of A, B, C, or D.

Reasoning:
"""

    return prompt

In [15]:
#Few-shot Prompt
def build_fewshot_prompt(example):

    fewshot = """
Question:
A patient with crushing chest pain radiating to the left arm most likely has:

Options:
A. Pneumonia
B. Myocardial infarction
C. GERD
D. Panic attack

Answer: B
"""

    question = example["question"]
    choices = get_choices(example)

    options_text = "\n".join(
        [f"{chr(65+i)}. {choice}" for i,choice in enumerate(choices)]
    )

    prompt = f"""
You are a medical expert solving USMLE questions.

Example:
{fewshot}

Now answer the following question.

Question:
{question}

Options:
{options_text}

Answer with only the letter (A, B, C, or D).

Answer:
"""

    return prompt

In [16]:
def run_inference_batch(data, prompt_builder, extractor, batch_size=8):

    correct = 0
    results = []

    prompts = [prompt_builder(data[i]) for i in range(len(data))]

    for i in tqdm(range(0, len(prompts), batch_size)):

        batch_prompts = prompts[i:i+batch_size]

        # HuggingFace dataset slice 문제 해결
        batch_examples = [
            data[k] for k in range(i, min(i+batch_size, len(data)))
        ]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

        for j in range(len(batch_prompts)):

            input_len = inputs["input_ids"].shape[1]

            generated_tokens = outputs[j][input_len:]

            decoded = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            )

            prediction = extractor(decoded)

            gold = get_gold_letter(batch_examples[j])

            is_correct = prediction == gold

            if is_correct:
                correct += 1

            results.append({
                "question": batch_examples[j]["question"],
                "gold": gold,
                "prediction": prediction,
                "correct": is_correct,
                "model_output": decoded
            })

    accuracy = correct / len(data)

    return accuracy, results

In [17]:
example = test_data[0]

print(get_choices(example))
print(get_gold_letter(example))

['Disclose the error to the patient and put it in the operative report', 'Tell the attending that he cannot fail to disclose this mistake', 'Report the physician to the ethics committee', 'Refuse to dictate the operative report']
B


In [18]:
example = test_data[0]

print(build_zero_shot_prompt(example))


You are a medical expert solving a USMLE clinical question.

Question:
A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?

Options:
A. Disclose the error to the patient and put it in the operative report
B. Tell the attending that he cannot fail to disclose this mistake
C. Report the physician to the ethics committee
D. Refuse to dictate the operative report

Choose the correct answer.

Answer:



In [19]:
example = test_data[0]

prompt = build_cot_prompt(example)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=False
)

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(decoded)
print("Prediction:", extract_cot_answer(decoded))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



You are a medical expert answering a USMLE clinical question.

Question:
A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?

Options:
A. Disclose the error to the patient and put it in the operative report
B. Tell the attending that he cannot fail to disclose this mistake
C. Report the physician to the ethics committee
D. Refuse to dictate the operative report

Think through the problem step by step.

At the end of your reasoning, output

In [20]:
# Zero-shot sample test
sample = test_data.select(range(20))

save_path = f"{DEBUG_RESULT_DIR}/MedQA_zeroshot_benchmark_20.json"

acc_sample, res_sample = run_inference_batch(
    sample,
    build_zero_shot_prompt,
    extract_choice
)

print("Zero-shot Sample Accuracy:", acc_sample)

100%|█████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:11<00:00,  3.88s/it]

Zero-shot Sample Accuracy: 0.5


In [21]:
# CoT sample test
sample = test_data.select(range(20))

save_path = f"{DEBUG_RESULT_DIR}/MedQA_cot_benchmark_20.json"

acc_sample, res_sample = run_inference_batch(
    sample,
    build_cot_prompt,
    extract_cot_answer
)

print("CoT Sample Accuracy:", acc_sample)

100%|█████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:11<00:00,  3.84s/it]

CoT Sample Accuracy: 0.35


In [22]:
# Few-shot sample test
sample = test_data.select(range(20))

save_path = f"{DEBUG_RESULT_DIR}/MedQA_fewshot_benchmark_20.json"

acc_sample, res_sample = run_inference_batch(
    sample,
    build_fewshot_prompt,
    extract_choice
)

print("Few-shot Sample Accuracy:", acc_sample)

100%|█████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:11<00:00,  3.88s/it]

Few-shot Sample Accuracy: 0.45


In [23]:
res_sample[:5]

[{'question': 'A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?',
  'gold': 'B',
  'prediction': 'A',
  'correct': False,
  'model_output': 'A.'},
 {'question': "A 67-year-old man with transitional cell carcinoma of the bladder comes to the physician because of a 2-day history of ringing sensation in his ear. He received this first course of neoadjuvant chemotherapy 1 week ago. Pure tone audiometry shows a sensorineural hearing loss of 

In [24]:
# Zero-shot full test
acc_full, res_full = run_inference_batch(
    test_data,
    build_zero_shot_prompt,
    extract_choice
)

print("Zero-shot Full Accuracy:", acc_full)

save_path = f"{BASELINE_RESULT_DIR}/MedQA_zeroshot_chat_full.json"

with open(save_path, "w") as f:
    json.dump(res_full, f, indent=2)

print("Saved:", save_path)

100%|█████████████████████████████████████████████████████████████████████████████████| 160/160 [10:37<00:00,  3.98s/it]

Zero-shot Full Accuracy: 0.6095836606441477
Saved: /gpfs/data/oermannlab/gaifl/users/yl14814/SLM_hybrid_project/03_results/baselines/llama31/MedQA/MedQA_zeroshot_chat_full.json


In [25]:
import numpy as np

print("Accuracy:", sum(r["correct"] for r in res_full)/len(res_full))

print("Average output length:",
      np.mean([len(r["model_output"]) for r in res_full]))

Accuracy: 0.6095836606441477
Average output length: 709.9104477611941


In [26]:
for r in res_full[:10]:
    print("MODEL OUTPUT:", r["model_output"])
    print("PRED:", r["prediction"])
    print("GOLD:", r["gold"])
    print()

MODEL OUTPUT: A. Disclose the error to the patient and put it in the operative report

Explanation:
The correct answer is A. Disclose the error to the patient and put it in the operative report. The resident has a duty to inform the patient of any complications that occurred during the procedure, even if the attending physician does not want to disclose this information. The patient has the right to know about any complications that may have occurred during their care. The resident should also document the complication in the operative report, as this is a permanent record of the case. This will also help to prevent similar mistakes from happening in the future.

The other options are incorrect because:

B. Telling the attending that he cannot fail to disclose this mistake may not be effective
PRED: A
GOLD: B

MODEL OUTPUT: B. Hyperstabilization of microtubules

Explanation:
The patient's symptoms are most likely due to cisplatin-induced ototoxicity. Cisplatin is a platinum-based chemo

In [27]:
from collections import Counter

print("Prediction distribution:")
print(Counter([r["prediction"] for r in res_full]))

Prediction distribution:
Counter({'C': 375, 'B': 351, 'A': 292, 'D': 233, 'INVALID': 11, 'c': 4, 'b': 4, 'a': 2, 'd': 1})


In [28]:
# CoT full test
acc_full, res_full = run_inference_batch(
    test_data,
    build_cot_prompt,
    extract_cot_answer
)

print("CoT Full Accuracy:", acc_full)

save_path = f"{BASELINE_RESULT_DIR}/MedQA_cot_chat_full.json"

with open(save_path, "w") as f:
    json.dump(res_full, f, indent=2)

print("Saved:", save_path)

100%|█████████████████████████████████████████████████████████████████████████████████| 160/160 [10:57<00:00,  4.11s/it]

CoT Full Accuracy: 0.401413982717989
Saved: /gpfs/data/oermannlab/gaifl/users/yl14814/SLM_hybrid_project/03_results/baselines/llama31/MedQA/MedQA_cot_chat_full.json


In [29]:
invalid_count = sum(1 for r in res_full if r["prediction"] == "INVALID")
print("INVALID predictions:", invalid_count)
print("Total:", len(res_full))

INVALID predictions: 21
Total: 1273


In [30]:
# Few-shot full test
acc_full, res_full = run_inference_batch(
    test_data,
    build_fewshot_prompt,
    extract_choice
)

print("Few-shot Full Accuracy:", acc_full)

save_path = f"{BASELINE_RESULT_DIR}/MedQA_fewshot_chat_full.json"

with open(save_path, "w") as f:
    json.dump(res_full, f, indent=2)

print("Saved:", save_path)

100%|█████████████████████████████████████████████████████████████████████████████████| 160/160 [11:04<00:00,  4.15s/it]

Few-shot Full Accuracy: 0.6040848389630793
Saved: /gpfs/data/oermannlab/gaifl/users/yl14814/SLM_hybrid_project/03_results/baselines/llama31/MedQA/MedQA_fewshot_chat_full.json


In [31]:
# analyze_result.py: 결과 분석
# error_analysis.py: 에러 분석
# cross-model error analysis: 3모델 비교

#1.analyze_result.py 역할: 해당 4개를 자동으로 생성
#→ accuracy table
#→ heatmap
#→ model ranking
#→ dataset ranking

#2.error_analysis.py 역할: hybrid 설계 근거 및 동시에 해당 3개의 파일 생성
#2-1. all_models_wrong 파일 근거:
#→ frontier model 필요
#→ retrieval 필요
#2-2. only_cot_correct 파일 근거:
#→ reasoning task
#2-3. only_fewshot_correct 파일 근거:
#→ pattern learning task

#3.cross-model error analysis 파일 근거: Hybrid routing 모델 설계에 사용, 각각의 모델만 맞춘 문제를 파악 및 3 모델 비교 분석을 자동 생성
#→ 3모델이 어떤 문제를 맞추고 틀렸는지 파악

#총단계: baseline 실행 → 결과 분석 → 에러 분석 → 모델 비교